# BarbellOS: ilustracje techniki z pozy manekina (Qwen-Image-Edit)

Gemini nie trzyma się techniki, więc pozę i sprzęt dyktuje kod: `scripts/technique_pose.py` rysuje szare manekiny (start i koniec ruchu, stopy w tym samym miejscu). Model **Qwen-Image-Edit-2511** (otwarty, Apache 2.0) maluje na nich naszą postać z `postac.png`.

**Zanim uruchomisz:** **Środowisko wykonawcze → Zmień typ środowiska wykonawczego → L4** (albo A100, NIE T4) i włącz **Duża ilość pamięci RAM**. Potem **Środowisko wykonawcze → Uruchom wszystko**.

Pierwsze uruchomienie pobiera model (ok. 58 GB, przy szybkim łączu Colaba kilka minut). Na końcu pobierze się `wyniki.zip`.

## 1. Ćwiczenia i ustawienia

In [ ]:
# Ćwiczenia do wygenerowania (szkice muszą być w scripts/colab/szkice w repozytorium).
EXERCISES = ["martwy"]
# Kilka wersji z różnym ziarnem losowania; wybierzemy najlepszą.
SEEDS = [1, 2]
STEPS = 40

## 2. Karta graficzna

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import psutil, torch
print(f"RAM: {psutil.virtual_memory().total / 1e9:.0f} GB")
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if gpu_gb < 20 or not torch.cuda.is_bf16_supported():
    raise SystemExit(
        "Ta karta jest za słaba (np. T4). Środowisko wykonawcze → Zmień typ środowiska wykonawczego "
        "→ L4 albo A100, potem Uruchom wszystko."
    )

## 3. Instalacja (diffusers z gałęzi głównej, jak każe karta modelu)

In [ ]:
!pip -q install -U git+https://github.com/huggingface/diffusers transformers accelerate bitsandbytes

## 4. Postać i szkice z repozytorium BarbellOS

Obrazek 1 to postać i styl, obrazek 2 to manekin w pozie.

In [ ]:
from diffusers.utils import load_image
from PIL import Image

BASE = "https://raw.githubusercontent.com/Preeksik/BarbellOS/main/scripts/colab/szkice"
character = load_image(f"{BASE}/postac.png").convert("RGB")
sketches = {
    name: [load_image(f"{BASE}/{name}-{i}.png").convert("RGB") for i in (1, 2)] for name in EXERCISES
}

def row(images, h=320):
    # Obrazki obok siebie, do podglądu.
    scaled = [im.resize((round(im.width * h / im.height), h)) for im in images]
    out = Image.new("RGB", (sum(im.width for im in scaled), h), "white")
    x = 0
    for im in scaled:
        out.paste(im, (x, 0))
        x += im.width
    return out

display(row([character] + [im for pair in sketches.values() for im in pair]))

## 5. Model: Qwen-Image-Edit-2511

Część rysująca (20 mld parametrów) jest skwantyzowana do 4 bitów (NF4), żeby zmieściła się na L4; części modelu idą na kartę na czas liczenia (`enable_model_cpu_offload`). Zajmie ok. 30 GB RAM.

In [ ]:
import gc
from diffusers import QwenImageEditPlusPipeline
from diffusers.quantizers import PipelineQuantizationConfig

quant = PipelineQuantizationConfig(
    quant_backend="bitsandbytes_4bit",
    quant_kwargs={"load_in_4bit": True, "bnb_4bit_quant_type": "nf4", "bnb_4bit_compute_dtype": torch.bfloat16},
    components_to_quantize=["transformer"],
)
pipe = QwenImageEditPlusPipeline.from_pretrained(
    "Qwen/Qwen-Image-Edit-2511", quantization_config=quant, torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()
pipe.set_progress_bar_config(disable=None)
gc.collect()
print(f"RAM zajęty: {psutil.virtual_memory().used / 1e9:.0f} GB, model gotowy")

## 6. Opisy ćwiczeń

Model dostaje pozę z manekina, a w opisie to, czego manekin nie mówi: styl, mięśnie na niebiesko i najważniejsze punkty techniki.

In [ ]:
STYLE = (
    "same face and hair, anatomical ecorche muscle illustration style with grey muscles, black shorts, "
    "black shoes, pure white background, soft studio light, side view, he faces left"
)
SPEC = {
    "martwy": {
        "muscles": "glutes, hamstrings, lower back erectors and upper trapezius",
        "start": "conventional deadlift start: the two black plates REST ON THE FLOOR, the bar is over the middle "
                 "of his foot and touches his shins, the near plate hides his shins, straight arms, hips higher "
                 "than the knees, back flat",
        "end": "deadlift lockout: standing fully upright on the same spot, hips and knees extended, bar in straight "
               "arms at the top of the thighs, no leaning back",
    },
    "rdl": {
        "muscles": "glutes, hamstrings and lower back erectors",
        "start": "Romanian deadlift top: standing upright, bar in straight arms at the top of the thighs, "
                 "knees slightly bent",
        "end": "Romanian deadlift bottom: hips pushed far back, knees slightly bent, back flat, bar at mid-shin, "
               "plates NOT touching the floor",
    },
}
NEGATIVE = (
    "grey mannequin, cartoon shapes, extra limbs, extra fingers, plates floating above the floor, bar in front "
    "of the feet, rounded back, text, numbers, logos, watermark, floor line, blurry, low quality"
)

def start_prompt(spec):
    return (
        f"Draw the man from image 1 ({STYLE}) in EXACTLY the pose, position and size of the grey mannequin in "
        f"image 2: same joint angles, same feet position on the floor, same barbell and plates position. "
        f"{spec['start']}. Muscles in blue (#2F6FD6) ONLY on {spec['muscles']}; all other muscles grey. "
        f"No text, no logos. The result must not look like a mannequin."
    )

def end_prompt(spec):
    return (
        f"Keep the man from image 1 exactly: same face, style, colors, camera, scale and the same position of his "
        f"feet on the floor. Change only his pose and the barbell to match the grey mannequin in image 2: "
        f"{spec['end']}. Muscles in blue (#2F6FD6) ONLY on {spec['muscles']}; all other muscles grey. "
        f"The result must not look like a mannequin."
    )

## 7. Generowanie

Dla każdego ziarna: najpierw start (postać + manekin startu), potem koniec ruchu z gotowego startu (żeby postać i stopy zostały te same). Czas jednego obrazka widać na pasku postępu.

In [ ]:
import os, time

os.makedirs("/content/wyniki", exist_ok=True)

def edit(images, prompt, seed):
    t = time.time()
    with torch.inference_mode():
        image = pipe(
            image=images, prompt=prompt, negative_prompt=NEGATIVE, true_cfg_scale=4.0, guidance_scale=1.0,
            num_inference_steps=STEPS, height=1024, width=1024,
            generator=torch.Generator(device="cpu").manual_seed(seed),
        ).images[0]
    print(f"  {time.time() - t:.0f} s")
    return image

for name in EXERCISES:
    spec, (sketch_start, sketch_end) = SPEC[name], sketches[name]
    for seed in SEEDS:
        print(name, "ziarno", seed, "start")
        start = edit([character, sketch_start], start_prompt(spec), seed)
        print(name, "ziarno", seed, "koniec ruchu")
        end = edit([start, sketch_end], end_prompt(spec), seed)
        start.save(f"/content/wyniki/{name}-s{seed}-1.png")
        end.save(f"/content/wyniki/{name}-s{seed}-2.png")
        display(row([sketch_start, start, sketch_end, end]))

## 8. Pobranie

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/wyniki", "zip", "/content/wyniki")
files.download("/content/wyniki.zip")

## Co sprawdzić

- Czy **talerze stoją na podłodze** na starcie martwego, a gryf jest przy piszczelach.
- Czy **stopy są w tym samym miejscu** na obu obrazkach.
- Czy to wciąż **nasza postać** (styl jak w `postac.png`), a nie szary manekin.
- Czy niebieskie są tylko właściwe mięśnie.

Rozpakuj `wyniki.zip` do `exports\technika\qwen\` i napisz agentowi, które ziarno lepsze (albo niech oceni sam).